# Python in Proteomics — code from the 2019 PeerJ preprint

**Source:** Hannes L. Röst, *Python in Proteomics*, PeerJ Preprints 27736v1 (2019).

This notebook collects the Python examples published in the preprint and turns them into a runnable Jupyter workflow using the modern `pyopenms` API where small compatibility changes are necessary.

The original paper demonstrates:

1. Reading mzML and FASTA files
2. Spectrum/chromatogram plotting
3. Elements and isotope distributions
4. Molecular formulas
5. Amino-acid sequences and peptide mass calculations
6. Modified peptides
7. FASTA proteins
8. Theoretical MS/MS spectra
9. Filtering spectra and peaks
10. Memory-efficient mzML processing
11. Simple peptide-spectrum searching

The preprint is available under CC BY 4.0. The examples below are derived from the code published in the preprint. 

## 1. Installation

The original 2019 manuscript used Python 3.7 and `pip install pyopenms`. The notebook below uses the current `pyopenms` package instead.

> **Important:** The paper is from 2019, so some APIs and plotting behavior have changed. Cells explicitly marked **modernized** preserve the scientific purpose of the original code while using current pyOpenMS conventions.

In [ ]:
%pip install -q -U pyopenms matplotlib pandas


In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyopenms as oms

print("Python:", sys.version.split()[0])
print("pyOpenMS:", getattr(oms, "__version__", "version unavailable"))


## 2. Download the example data used by the paper

The preprint points to the Zenodo record `2653155` and downloads:

- `example.mzML`
- `search.fasta`

The original code used `https://zenodo.org/record/2653155/files/`. Zenodo's URL structure has changed since 2019, so this notebook tries the modern `/records/` URL first and then the original URL.

If the historical record is unavailable, put `example.mzML` and `search.fasta` in the notebook directory and rerun the loading cells.

In [ ]:
from urllib.request import urlretrieve
from urllib.error import URLError, HTTPError

DATA_DIR = Path("pyopenms_paper_data")
DATA_DIR.mkdir(exist_ok=True)

files = {
    "example.mzML": [
        "https://zenodo.org/records/2653155/files/example.mzML?download=1",
        "https://zenodo.org/record/2653155/files/example.mzML",
    ],
    "search.fasta": [
        "https://zenodo.org/records/2653155/files/search.fasta?download=1",
        "https://zenodo.org/record/2653155/files/search.fasta",
    ],
}

def download_first_available(filename, urls):
    target = DATA_DIR / filename
    if target.exists():
        print(f"Already present: {target}")
        return target

    for url in urls:
        try:
            print("Trying:", url)
            urlretrieve(url, target)
            print("Downloaded:", target)
            return target
        except Exception as exc:
            print("  failed:", type(exc).__name__, exc)

    print(f"Could not download {filename}.")
    print(f"Please place it manually at: {target.resolve()}")
    return target

mzml_path = download_first_available("example.mzML", files["example.mzML"])
fasta_path = download_first_available("search.fasta", files["search.fasta"])


## 3. Read an mzML file

This is the first major example in the preprint. The paper loads the file into an `MSExperiment` and reports the number of spectra and chromatograms.

In [ ]:
exp = oms.MSExperiment()

if mzml_path.exists():
    oms.MzMLFile().load(str(mzml_path), exp)
    print("Number of spectra:", exp.getNrSpectra())
    print("Number of chromatograms:", exp.getNrChromatograms())
else:
    print("example.mzML is not available yet.")


## 4. Plot a spectrum and chromatogram

The preprint used:

```python
Plot.plotSpectrum(exp.getSpectrum(0))
Plot.plotChromatogram(exp.getChromatogram(0))
```

That plotting interface is historical. The modernized cells below extract the same pyOpenMS data and use matplotlib, which is more robust in a Jupyter environment.

In [ ]:
if exp.getNrSpectra() > 0:
    spectrum = exp.getSpectrum(0)
    mz, intensity = spectrum.get_peaks()

    plt.figure(figsize=(10, 4))
    plt.vlines(mz, 0, intensity)
    plt.xlabel("m/z")
    plt.ylabel("Intensity")
    plt.title("Spectrum 0")
    plt.tight_layout()
    plt.show()


In [ ]:
if exp.getNrChromatograms() > 0:
    chromatogram = exp.getChromatogram(0)
    rt, intensity = chromatogram.get_peaks()

    plt.figure(figsize=(10, 4))
    plt.plot(rt, intensity)
    plt.xlabel("Retention time")
    plt.ylabel("Intensity")
    plt.title("Chromatogram 0")
    plt.tight_layout()
    plt.show()


## 5. Elements and isotope distributions

This reproduces the paper's sulfur example using `ElementDB`.

In [ ]:
edb = oms.ElementDB()
sulfur = edb.getElement("S")

print("Element:", sulfur.getName())
print("Symbol:", sulfur.getSymbol())

isotopes = sulfur.getIsotopeDistribution()

for iso in isotopes.getContainer():
    print(f"m/z = {iso.getMZ():.8f}, intensity = {iso.getIntensity():.8f}")


## 6. Molecular formulas

The paper demonstrates construction and arithmetic with `EmpiricalFormula` objects.

In [ ]:
methanol = oms.EmpiricalFormula("CH3OH")
water = oms.EmpiricalFormula("H2O")
ethanol = oms.EmpiricalFormula("CH2" + methanol.toString())

wm = water + methanol

print("Methanol:", methanol)
print("Water:", water)
print("Ethanol:", ethanol)
print("Water + methanol:", wm)
print("Elemental composition:", wm.getElementalComposition())


## 7. Coarse and fine isotope distributions

The preprint demonstrates both `CoarseIsotopePatternGenerator` and `FineIsotopePatternGenerator`.

In [ ]:
wm = oms.EmpiricalFormula("CH3OH") + oms.EmpiricalFormula("H2O")

print("Coarse isotope distribution:")
coarse = wm.getIsotopeDistribution(
    oms.CoarseIsotopePatternGenerator(5)
)

for iso in coarse.getContainer():
    print(f"{iso.getMZ():.8f} : {iso.getIntensity():.8f}")

print("\nFine isotope distribution:")
fine = wm.getIsotopeDistribution(
    oms.FineIsotopePatternGenerator(1e-5)
)

for iso in fine.getContainer():
    print(f"{iso.getMZ():.8f} : {iso.getIntensity():.8f}")


## 8. Amino-acid sequences

The paper uses `AASequence` with the peptide `DFPIANGER`. This section follows the example and adds the prefix/suffix display used in current pyOpenMS documentation.

In [ ]:
seq = oms.AASequence.fromString("DFPIANGER")

prefix = seq.getPrefix(4)
suffix = seq.getSuffix(5)
concat = seq + seq

print("Sequence:", seq)
print("Prefix:", prefix)
print("Suffix:", suffix)
print("Concatenated:", concat)

m_full = seq.getMonoWeight()
m_precursor = seq.getMonoWeight(oms.Residue.ResidueType.Full, 2)
mz_precursor = m_precursor / 2.0

print()
print("Monoisotopic mass [M]:", m_full)
print("Monoisotopic mass [M+2H]2+:", m_precursor)
print("Monoisotopic m/z [M+2H]2+:", mz_precursor)


In [ ]:
print("Amino acids in the peptide:")
for aa in seq:
    print(f"{aa.getName():15s} : {aa.getMonoWeight():.10f}")


## 9. Peptide formula, isotope pattern and fragment ions

This is the molecular-formula/fragment-ion example from the preprint.

In [ ]:
seq = oms.AASequence.fromString("DFPIANGER")
seq_formula = seq.getFormula()

print("Peptide", seq, "has molecular formula", seq_formula)
print("=" * 35)

gen = oms.CoarseIsotopePatternGenerator(6)
isotopes = seq_formula.getIsotopeDistribution(gen)

for iso in isotopes.getContainer():
    print("Isotope", iso.getMZ(), ":", iso.getIntensity())

suffix = seq.getSuffix(3)  # y3 ion "GER"

print("=" * 35)
print("y3 ion:", suffix)

y3_formula = suffix.getFormula(oms.Residue.ResidueType.YIon, 2)
y3_mz = suffix.getMonoWeight(oms.Residue.ResidueType.YIon, 2) / 2.0

print("y3 m/z:", y3_mz)
print("y3 formula:", y3_formula)
print("Peptide formula:", seq_formula)


## 10. Modified peptide sequences

The preprint demonstrates several equivalent ways of representing an oxidized methionine and mass-based modifications.

In [ ]:
modified = oms.AASequence.fromString(
    "PEPTIDESEKUEM(Oxidation)CER"
)

print("Unmodified:", modified.toUnmodifiedString())
print("OpenMS string:", modified.toString())
print("UniMod string:", modified.toUniModString())
print("Bracket string:", modified.toBracketString())
print("Bracket string without names:", modified.toBracketString(False))

examples = [
    "DFPIAM(UniMod:35)GER",
    "DFPIAM[+16]GER",
    "DFPIAM[+15.99]GER",
    "DFPIAM[147]GER",
    "DFPIAM[147.035405]GER",
]

for text in examples:
    try:
        print(text, "->", oms.AASequence.fromString(text))
    except Exception as exc:
        print(text, "-> ERROR:", exc)


## 11. FASTA proteins

The preprint constructs two protein entries and writes them to a FASTA file.

In [ ]:
bsa = oms.FASTAEntry()
bsa.sequence = "MKWVTFISLLLLFSSAYSRGVFRRDTHKSEIAHRFKDLGE"
bsa.description = "BSA Bovine Albumin (partial sequence)"
bsa.identifier = "BSA"

alb = oms.FASTAEntry()
alb.sequence = "MKWVTFISLLFLFSSAYSRGVFRRDAHKSEVAHRFKDLGE"
alb.description = "ALB Human Albumin (partial sequence)"
alb.identifier = "ALB"

entries = [bsa, alb]

output_fasta = DATA_DIR / "example_generated.fasta"
oms.FASTAFile().store(str(output_fasta), entries)

print("Wrote:", output_fasta)
print(output_fasta.read_text())


## 12. Generate theoretical MS/MS spectra

This reproduces the paper's `TheoreticalSpectrumGenerator` example.

In [ ]:
tsg = oms.TheoreticalSpectrumGenerator()

spec1 = oms.MSSpectrum()
spec2 = oms.MSSpectrum()

peptide = oms.AASequence.fromString("DFPIANGER")

# Standard behavior: b- and y-ions of charge 1.
p = oms.Param()
p.setValue("add_b_ions", "false")
tsg.setParameters(p)

tsg.getSpectrum(spec1, peptide, 1, 1)

# Add a-, b-, y-ions, losses and annotation metadata.
p.setValue("add_b_ions", "true")
p.setValue("add_a_ions", "true")
p.setValue("add_losses", "true")
p.setValue("add_metainfo", "true")
tsg.setParameters(p)

tsg.getSpectrum(spec2, peptide, 1, 2)

print("Spectrum 1 has", spec1.size(), "peaks.")
print("Spectrum 2 has", spec2.size(), "peaks.")


In [ ]:
# Inspect annotated ions and masses.
arrays = spec2.getStringDataArrays()

if arrays:
    annotations = arrays[0]
    for ion, peak in zip(annotations, spec2):
        print(ion, peak.getMZ())
else:
    print("No annotation array was returned.")


In [ ]:
# Plot the theoretical spectrum.
mz_theoretical, intensity_theoretical = spec2.get_peaks()

plt.figure(figsize=(12, 4))
plt.vlines(mz_theoretical, 0, intensity_theoretical)
plt.xlabel("m/z")
plt.ylabel("Relative intensity")
plt.title("Theoretical MS/MS spectrum: DFPIANGER")
plt.tight_layout()
plt.show()


## 13. Filtering spectra by native ID

This follows the paper's first spectrum-filtering example.

In [ ]:
if mzml_path.exists():
    inp = oms.MSExperiment()
    oms.MzMLFile().load(str(mzml_path), inp)

    filtered = oms.MSExperiment()

    for s in inp:
        if s.getNativeID().startswith("scan="):
            filtered.addSpectrum(s)

    filtered_path = DATA_DIR / "test_filtered_native_id.mzML"
    oms.MzMLFile().store(str(filtered_path), filtered)

    print("Input spectra:", inp.getNrSpectra())
    print("Output spectra:", filtered.getNrSpectra())
    print("Written:", filtered_path)
else:
    print("example.mzML not available.")


## 14. Filter by MS level

The paper retains only MS2+ spectra.

In [ ]:
if mzml_path.exists():
    inp = oms.MSExperiment()
    oms.MzMLFile().load(str(mzml_path), inp)

    filtered_ms2 = oms.MSExperiment()

    for s in inp:
        if s.getMSLevel() > 1:
            filtered_ms2.addSpectrum(s)

    filtered_path = DATA_DIR / "test_filtered_ms2plus.mzML"
    oms.MzMLFile().store(str(filtered_path), filtered_ms2)

    print("Input spectra:", inp.getNrSpectra())
    print("MS2+ spectra:", filtered_ms2.getNrSpectra())
    print("Written:", filtered_path)


## 15. Filter by scan number

The preprint uses an external list of scan indices.

In [ ]:
if mzml_path.exists():
    inp = oms.MSExperiment()
    oms.MzMLFile().load(str(mzml_path), inp)

    scan_nrs = [0, 2, 5, 7]
    filtered_scans = oms.MSExperiment()

    for k, s in enumerate(inp):
        if k in scan_nrs and s.getMSLevel() == 1:
            filtered_scans.addSpectrum(s)

    filtered_path = DATA_DIR / "test_filtered_scan_numbers.mzML"
    oms.MzMLFile().store(str(filtered_path), filtered_scans)

    print("Selected spectra:", filtered_scans.getNrSpectra())
    print("Written:", filtered_path)


## 16. Filter spectra and peaks by m/z

The manuscript uses an example window of 6–12 m/z because the demonstration file is a small teaching dataset. In a real experiment, replace these values with a scientifically meaningful region.

In [ ]:
if mzml_path.exists():
    inp = oms.MSExperiment()
    oms.MzMLFile().load(str(mzml_path), inp)

    mz_start = 6.0
    mz_end = 12.0

    filtered = oms.MSExperiment()

    for s in inp:
        if s.getMSLevel() > 1:
            mz_values, intensities = s.get_peaks()

            keep = (mz_values > mz_start) & (mz_values < mz_end)
            s.set_peaks((mz_values[keep], intensities[keep]))

            filtered.addSpectrum(s)

    filtered_path = DATA_DIR / "test_filtered_peaks.mzML"
    oms.MzMLFile().store(str(filtered_path), filtered)

    print("Written:", filtered_path)


## 17. A more useful real-world peak filter

The same mechanism can be used to retain peaks above an intensity threshold or the top N peaks. This is a direct extension of the filtering example in the paper.

In [ ]:
def keep_top_n_peaks(spectrum, n=10):
    mz_values, intensities = spectrum.get_peaks()

    if len(intensities) <= n:
        return spectrum

    indices = np.argsort(intensities)[-n:]
    indices = np.sort(indices)

    spectrum.set_peaks((
        mz_values[indices],
        intensities[indices]
    ))
    return spectrum

if mzml_path.exists():
    inp = oms.MSExperiment()
    oms.MzMLFile().load(str(mzml_path), inp)

    top_n_exp = oms.MSExperiment()

    for s in inp:
        if s.getMSLevel() > 1:
            top_n_exp.addSpectrum(keep_top_n_peaks(s, n=10))

    print("Processed", top_n_exp.getNrSpectra(), "MS2+ spectra.")


## 18. Memory-efficient processing with `OnDiscMSExperiment`

The preprint introduces `OnDiscMSExperiment` to avoid loading the complete mzML into memory.

In [ ]:
if mzml_path.exists():
    ondisc_exp = oms.OnDiscMSExperiment()
    ondisc_exp.openFile(str(mzml_path))

    print("Spectra available:", ondisc_exp.getNrSpectra())

    for k in range(ondisc_exp.getNrSpectra()):
        s = ondisc_exp.getSpectrum(k)

        if s.getNativeID().startswith("scan="):
            # Work with the spectrum here without loading the entire
            # experiment into an in-memory MSExperiment.
            mz_values, intensities = s.get_peaks()
            print(
                "scan", k,
                "native ID:", s.getNativeID(),
                "peaks:", len(mz_values)
            )
            if k >= 2:
                break


## 19. Streaming output with `PlainMSDataWritingConsumer`

This is the memory-saving pattern described in the paper: read spectra on demand and immediately write selected spectra instead of accumulating them in an `MSExperiment`.

In [ ]:
if mzml_path.exists():
    ondisc_exp = oms.OnDiscMSExperiment()
    ondisc_exp.openFile(str(mzml_path))

    streaming_output = DATA_DIR / "test_filtered_streaming.mzML"
    consumer = oms.PlainMSDataWritingConsumer(str(streaming_output))

    for k in range(ondisc_exp.getNrSpectra()):
        s = ondisc_exp.getSpectrum(k)

        if s.getNativeID().startswith("scan="):
            consumer.consumeSpectrum(s)

    # The paper explicitly notes that the consumer must be deleted so
    # the final mzML content is flushed and closed.
    del consumer

    print("Written:", streaming_output)


## 20. Simple peptide search

The preprint uses `SimpleSearchEngineAlgorithm` as a teaching example. It searches the experimental mzML against a FASTA database.

This is intentionally kept optional because it requires the paper's example files and the corresponding search-engine support in the installed OpenMS version.

In [ ]:
if mzml_path.exists() and fasta_path.exists():
    protein_ids = []
    peptide_ids = []

    try:
        oms.SimpleSearchEngineAlgorithm().search(
            str(mzml_path),
            str(fasta_path),
            protein_ids,
            peptide_ids,
        )

        print("Protein IDs:", len(protein_ids))
        print("Peptide IDs:", len(peptide_ids))

    except Exception as exc:
        print("Search could not be executed:", type(exc).__name__, exc)
else:
    print("Both example.mzML and search.fasta are required.")


## 21. Inspect peptide-spectrum matches

This reproduces the PSM inspection logic from the paper, with defensive handling for metadata that may differ between OpenMS versions.

In [ ]:
for peptide_id in peptide_ids:
    print("=" * 60)
    print("Peptide ID m/z:", peptide_id.getMZ())
    print("Peptide ID RT:", peptide_id.getRT())
    print("Peptide scan index:", peptide_id.getMetaValue("scan_index")
          if peptide_id.metaValueExists("scan_index") else "NA")
    print("Peptide ID score type:", peptide_id.getScoreType())

    for hit in peptide_id.getHits():
        print(" - rank:", hit.getRank())
        print(" - charge:", hit.getCharge())
        print(" - sequence:", hit.getSequence())

        z = hit.getCharge()
        mz = (
            hit.getSequence().getMonoWeight(
                oms.Residue.ResidueType.Full, z
            ) / z
        )

        print(" - monoisotopic m/z:", mz)
        print(
            " - ppm error:",
            abs(mz - peptide_id.getMZ()) / mz * 1e6
        )
        print(" - score:", hit.getScore())


## 22. Generate the theoretical spectrum for the peptide identified in the paper

The preprint investigates:

`RPGADSDIGGFGGLFDLAQAGFR`

and compares its theoretical fragment ions with an experimental spectrum.

In [ ]:
target_peptide = oms.AASequence.fromString(
    "RPGADSDIGGFGGLFDLAQAGFR"
)

tsg = oms.TheoreticalSpectrumGenerator()
thspec = oms.MSSpectrum()

p = oms.Param()
p.setValue("add_metainfo", "true")
tsg.setParameters(p)

tsg.getSpectrum(
    thspec,
    target_peptide,
    1,
    1
)

print("Theoretical peaks:", thspec.size())

arrays = thspec.getStringDataArrays()
if arrays:
    annotations = arrays[0]
    for ion, peak in zip(annotations, thspec):
        print(ion, peak.getMZ())


## 23. Compare theoretical and experimental spectra

The original manuscript loads `searchfile.mzML`, takes spectrum index 2, and retains peaks above intensity 1500 and m/z 300.

Here we make that operation work with the downloaded example file if the expected spectrum exists.

In [ ]:
if mzml_path.exists():
    e = oms.MSExperiment()
    oms.MzMLFile().load(str(mzml_path), e)

    if e.getNrSpectra() > 2:
        spectrum = e[2]

        print("Spectrum native ID:", spectrum.getNativeID())

        mz_values, intensities = spectrum.get_peaks()

        mask = (intensities > 1500) & (mz_values > 300)

        exp_mz = mz_values[mask]
        exp_intensity = intensities[mask]

        print("Filtered experimental peaks:", len(exp_mz))

        plt.figure(figsize=(12, 4))
        plt.vlines(exp_mz, 0, exp_intensity, label="Experimental")
        plt.xlabel("m/z")
        plt.ylabel("Intensity")
        plt.title("Experimental spectrum — filtered peaks")
        plt.tight_layout()
        plt.show()
    else:
        print("The example file contains fewer than 3 spectra.")


## 24. Plot theoretical and experimental spectra together

This is a notebook-friendly replacement for the TOPPView comparison shown in the paper.

In [ ]:
if mzml_path.exists() and e.getNrSpectra() > 2:
    exp_spectrum = e[2]
    exp_mz, exp_intensity = exp_spectrum.get_peaks()

    plt.figure(figsize=(14, 6))

    # Experimental peaks
    plt.vlines(
        exp_mz,
        0,
        exp_intensity / max(exp_intensity) if len(exp_intensity) else exp_intensity,
        linewidth=1,
        label="Experimental"
    )

    # Theoretical peaks
    th_mz, th_intensity = thspec.get_peaks()
    if len(th_intensity):
        th_norm = th_intensity / max(th_intensity)
        plt.vlines(
            th_mz,
            0,
            -th_norm,
            linewidth=1,
            label="Theoretical"
        )

    plt.axhline(0, linewidth=0.8)
    plt.xlabel("m/z")
    plt.ylabel("Normalized intensity")
    plt.title("Experimental vs theoretical MS/MS spectrum")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 25. Compact reusable functions

The following turns the main ideas from the preprint into functions that can be reused in your own proteomics notebooks.

In [ ]:
def load_mzml(path):
    exp = oms.MSExperiment()
    oms.MzMLFile().load(str(path), exp)
    return exp


def spectrum_to_dataframe(spectrum):
    mz, intensity = spectrum.get_peaks()
    return pd.DataFrame({
        "mz": mz,
        "intensity": intensity,
    })


def chromatogram_to_dataframe(chromatogram):
    rt, intensity = chromatogram.get_peaks()
    return pd.DataFrame({
        "rt": rt,
        "intensity": intensity,
    })


def generate_theoretical_spectrum(sequence, min_charge=1, max_charge=2):
    peptide = oms.AASequence.fromString(sequence)
    spectrum = oms.MSSpectrum()

    generator = oms.TheoreticalSpectrumGenerator()
    params = oms.Param()
    params.setValue("add_metainfo", "true")
    generator.setParameters(params)

    generator.getSpectrum(
        spectrum,
        peptide,
        min_charge,
        max_charge
    )
    return spectrum


# Example:
theoretical = generate_theoretical_spectrum("DFPIANGER")
df = spectrum_to_dataframe(theoretical)
df.head()


## 26. Summary of the code retrieved from the preprint

| Preprint section | Main pyOpenMS functionality |
|---|---|
| Installation | `pyopenms` |
| File handling | `MSExperiment`, `MzMLFile`, `FASTAFile` |
| Plotting | spectra and chromatograms |
| Elements | `ElementDB` |
| Molecular formulas | `EmpiricalFormula` |
| Isotopes | coarse/fine isotope generators |
| Peptides | `AASequence`, `Residue` |
| Modifications | `ResidueModification` / modified `AASequence` |
| Proteins | `FASTAEntry`, `FASTAFile` |
| MS/MS generation | `TheoreticalSpectrumGenerator` |
| Filtering | MS level, native ID, scan index, m/z |
| Large files | `OnDiscMSExperiment` |
| Streaming | `PlainMSDataWritingConsumer` |
| Peptide search | `SimpleSearchEngineAlgorithm` |
| PSM inspection | `PeptideIdentification`, `PeptideHit` |

The R/`reticulate` section of the paper was not included because this is a Python/Jupyter notebook; all Python examples from the main technical sections are represented above.

## References

**Original preprint**

Röst, H. L. (2019). *Python in Proteomics*. PeerJ Preprints 27736v1. DOI: 10.7287/peerj.preprints.27736v1.

The preprint explicitly demonstrates pyOpenMS for mzML handling, visualization, chemical calculations, peptide/protein manipulation, theoretical spectra, data filtering, memory-efficient processing, and a teaching-oriented peptide search. 

**Modern pyOpenMS documentation**

The current pyOpenMS documentation retains and expands many of the peptide, formula, isotope, and spectrum-generation examples used by the preprint.
